# Imports

In [ ]:
import sys, os

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

if ROOT not in sys.path:
    sys.path.append(ROOT)

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
from modules.attachment_extractor import extract_attachment_text
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


# Load data + filter attachments

In [ ]:
df = pd.read_csv("../../datasets/final/final_combined.csv")

# Keep only attachments with at least 20 characters (remove noise)
attach_df = df[df["attachment_text"].astype(str).str.strip().str.len() > 20].copy()

print("Total usable attachment samples:", len(attach_df))

train_df, val_df = train_test_split(
    attach_df,
    test_size=0.1,
    random_state=42,
    stratify=attach_df["label"]
)

print("Train:", len(train_df), "Val:", len(val_df))

Total usable attachment samples: 0


/tmp/ipykernel_226687/2287213665.py:1: DtypeWarning: Columns (3,4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../datasets/processed/final_combined.csv")


ValueError: With n_samples=0, test_size=0.1 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

# Load XLM-RoBERTa-Large + tokenizer

In [ ]:
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure PAD token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# VRAM optimization
model.gradient_checkpointing_enable()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("DEVICE:", device)


# Convert to HuggingFace Datasets + tokenize

In [ ]:
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

def tokenize(batch):
    return tokenizer(
        batch["attachment_text"],
        truncation=True,
        padding="max_length",
        max_length=512   # attachments are long
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format("torch",   columns=["input_ids", "attention_mask", "labels"])


# Data collator

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)

# Metrics with sklearn

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )

    print("\nConfusion Matrix:")
    print(confusion_matrix(labels, preds))

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="attachment_model_xlmr_large_out",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=1e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.06,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch-size = 16
    
    num_train_epochs=3,
    weight_decay=0.01,

    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    optim="adamw_torch",
    label_smoothing_factor=0.1,
    report_to="none",
)


# Trainer + Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Save Model + Tokenizer

In [ ]:
trainer.save_model("attachment_encoder_xlmr_large")
tokenizer.save_pretrained("attachment_encoder_xlmr_large_tokenizer")

print("Training complete and model saved.")

# Confusion Matrix Plot

In [ ]:
preds = trainer.predict(val_ds)
y_true = preds.label_ids
y_pred = preds.predictions.argmax(-1)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Phishing"])
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - XLM-R Large Attachment Model")
plt.show()

# Plot Loss Curve

In [ ]:
logs = trainer.state.log_history
df_logs = pd.DataFrame(logs)

plt.figure(figsize=(10,5))
loss_df = df_logs[df_logs["loss"].notna()]
plt.plot(loss_df["step"], loss_df["loss"])
plt.title("Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Plot Metrics Curve

In [ ]:
eval_df = df_logs[df_logs["eval_loss"].notna()].copy()

if "epoch" not in eval_df.columns:
    eval_df["epoch"] = range(1, len(eval_df)+1)

plt.figure(figsize=(14,6))

for col, label in [
    ("eval_accuracy", "Accuracy"),
    ("eval_precision", "Precision"),
    ("eval_recall", "Recall"),
    ("eval_f1", "F1 Score")
]:
    if col in eval_df:
        plt.plot(eval_df["epoch"], eval_df[col], marker="o", label=label)

plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("Evaluation Metrics per Epoch (Attachment Model)")
plt.legend()
plt.grid(True)
plt.show()


# Final Precision-Recall Curve

In [ ]:
preds = trainer.predict(val_ds)
y_true = preds.label_ids

# Safer probability extraction
probs = torch.softmax(torch.tensor(preds.predictions), dim=1)[:, 1].numpy()

prec, rec, thresh = precision_recall_curve(y_true, probs)

plt.figure(figsize=(10,5))
plt.plot(rec, prec)
plt.title("Precision-Recall Curve")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()


# Final Classification Report

In [ ]:

final_preds = preds.predictions.argmax(-1)
print(classification_report(y_true, final_preds, target_names=["Legit", "Phishing"]))

# Inference Helper (User Input)

In [ ]:
def predict_attachment(text):
    model.eval()
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(-1).item()

    return {
        "prediction": "PHISHING" if pred==1 else "LEGIT",
        "prob_legit": float(probs[0][0]),
        "prob_phishing": float(probs[0][1])
    }

def predict_attachment_file(file_path):
    text = extract_attachment_text(file_path)
    if not text.strip():
        return {"error": "No extractable text from attachment."}

    return predict_attachment(text)  # reuse Stage-2B inference method

# Example:
print(predict_attachment_file("path/to/file"))
